# MGMT298D: Science and Strategy of AI
## Assignment 5 - Neural Network Architecture Tuning
### Application: Fashion Product Classification

---

**Instructions:** Complete the exercises by filling in the `???` placeholders and answering the questions. Run all code cells in order.

## Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

np.random.seed(42)
tf.random.set_seed(42)

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train_full = x_train_full.astype("float32").reshape(-1, 784) / 255.0
x_test = x_test.astype("float32").reshape(-1, 784) / 255.0

X_train, X_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)

print(f"Training: {X_train.shape} | Validation: {X_val.shape} | Test: {x_test.shape}")

## Part 1: Effect of Hidden Layer Size

Let's explore how the number of neurons in a hidden layer affects model performance.

In [ ]:
def build_model(hidden_units):
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(hidden_units, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# Test different hidden layer sizes
hidden_sizes = [4, 8, 16, 32, 64, 128]
results = {"hidden_units": [], "val_accuracy": [], "test_accuracy": [], "params": []}

for h in hidden_sizes:
    print(f"Training with {h} hidden units...")
    model = build_model(h)
    history = model.fit(X_train, y_train, epochs=20, batch_size=128, 
                        validation_data=(X_val, y_val), verbose=0)
    
    val_acc = max(history.history['val_accuracy'])
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    
    results["hidden_units"].append(h)
    results["val_accuracy"].append(val_acc)
    results["test_accuracy"].append(test_acc)
    results["params"].append(model.count_params())

results_df = pd.DataFrame(results)
print("\n=== Results ===")
print(results_df.to_string(index=False))

In [ ]:
# Visualize the results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(results_df["hidden_units"], results_df["test_accuracy"], marker="o", linewidth=2)
ax1.set_xlabel("Hidden Units")
ax1.set_ylabel("Test Accuracy")
ax1.set_title("Accuracy vs Hidden Layer Size")
ax1.grid(True)

ax2.plot(results_df["params"], results_df["test_accuracy"], marker="o", linewidth=2, color="orange")
ax2.set_xlabel("Number of Parameters")
ax2.set_ylabel("Test Accuracy")
ax2.set_title("Accuracy vs Model Complexity")
ax2.grid(True)

plt.tight_layout()
plt.show()

## Part 2: Effect of Dropout Rate

Dropout is a regularization technique. Let's see how different dropout rates affect overfitting.

In [ ]:
def build_model_with_dropout(dropout_rate):
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# ============================================================
# EXERCISE: Fill in the dropout rates to test
# ============================================================
dropout_rates = [0.0, 0.1, 0.3, 0.5, 0.7]  # <-- These are filled in for you

dropout_results = {"dropout": [], "train_acc": [], "val_acc": [], "gap": []}

for dr in dropout_rates:
    print(f"Training with dropout = {dr}...")
    model = build_model_with_dropout(dr)
    history = model.fit(X_train, y_train, epochs=30, batch_size=128,
                        validation_data=(X_val, y_val), verbose=0)
    
    train_acc = max(history.history['accuracy'])
    val_acc = max(history.history['val_accuracy'])
    
    dropout_results["dropout"].append(dr)
    dropout_results["train_acc"].append(train_acc)
    dropout_results["val_acc"].append(val_acc)
    dropout_results["gap"].append(train_acc - val_acc)

dropout_df = pd.DataFrame(dropout_results)
print("\n=== Dropout Results ===")
print(dropout_df.to_string(index=False))

In [ ]:
# Visualize overfitting gap
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(dropout_rates))
width = 0.35

ax.bar(x - width/2, dropout_df["train_acc"], width, label="Train", color="#4ecdc4")
ax.bar(x + width/2, dropout_df["val_acc"], width, label="Validation", color="#45b7d1")

ax.set_xlabel("Dropout Rate")
ax.set_ylabel("Accuracy")
ax.set_title("Train vs Validation Accuracy by Dropout Rate")
ax.set_xticks(x)
ax.set_xticklabels(dropout_rates)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSmallest train-val gap: dropout = {dropout_df.loc[dropout_df['gap'].idxmin(), 'dropout']}")

## Part 3: Build Your Best Model

Using what you learned, build a model with your chosen architecture.

In [ ]:
# ============================================================
# EXERCISE: Fill in your chosen hyperparameters
# ============================================================
HIDDEN_1 = ???  # First hidden layer size (e.g., 64, 128)
HIDDEN_2 = ???  # Second hidden layer size (e.g., 32, 64)
DROPOUT = ???   # Dropout rate (e.g., 0.2, 0.3)

best_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(HIDDEN_1, activation="relu"),
    layers.Dropout(DROPOUT),
    layers.Dense(HIDDEN_2, activation="relu"),
    layers.Dropout(DROPOUT),
    layers.Dense(10, activation="softmax")
])
best_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stopping = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)

history = best_model.fit(X_train, y_train, epochs=50, batch_size=128,
                         validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=1)

test_loss, test_acc = best_model.evaluate(x_test, y_test)
print(f"\nYour Model — Test Accuracy: {test_acc*100:.2f}%")

## Part 4: Error Analysis

In [ ]:
# Confusion matrix
y_pred = np.argmax(best_model.predict(x_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
acc_df = pd.DataFrame({"Class": CLASS_NAMES, "Accuracy": per_class_acc}).sort_values("Accuracy")
print("\nPer-Class Accuracy (sorted):")
print(acc_df.to_string(index=False))

---
## Questions

### Question 1: Hidden Layer Size

**Q1a:** Looking at your results, at what point did increasing hidden units stop improving accuracy significantly? Why does accuracy plateau?

*Your answer:*



**Q1b:** A colleague suggests "just use 1000 hidden units to be safe." What are the downsides of an overly large network, even if accuracy doesn't decrease?

*Your answer:*



---
### Question 2: Dropout and Overfitting

**Q2a:** Look at the train-val gap for different dropout rates. How does dropout reduce overfitting? What happened when dropout was too high (0.7)?

*Your answer:*



**Q2b:** In a business context, why is the gap between training and validation accuracy important? What risk does a large gap represent?

*Your answer:*



---
### Question 3: Error Analysis

**Q3:** Look at the confusion matrix. Which pairs of clothing items are most often confused? Why might the model struggle with these specific pairs, and how could a fashion retailer use this information?

*Your answer:*



---
### Question 4: Business Application

**Q4:** An e-commerce company wants to use this model to auto-tag product images. Given that the model is ~88% accurate, what guardrails or human-in-the-loop processes would you recommend before deploying it?

*Your answer:*



---
### Question 5: Neural Networks vs Traditional ML

**Q5:** In Week 5, we saw that XGBoost achieved ~78% accuracy on this dataset while neural networks reached ~88%. For image classification, why do neural networks outperform tree-based models? When might you still choose XGBoost over a neural network?

*Your answer:*

